# Merchant Risk & Underwriting

Build a **grounded, self-verifying** underwriting agent that scores new-merchant applications — approve / refer / decline, risk score, reserve % — and cross-references a **CFPB consumer-complaint backdrop** for payment products.

| Step | What you build |
|---|---|
| 1 | Explore the benchmark (`MERCHANT_RISK_BENCHMARK`) |
| 2 | Query the **semantic view** (Cortex Analyst) |
| 3 | Build a verified **fact pack** |
| 4 | **Grounded draft** under DATA RULES |
| 5 | **Verify** claims vs evidence |
| 6 | Persist the note |

**Prereq:** run `setup_synthetic.sql` (no listing) or `setup.sql` (real free CFPB Complaint Database) as a role that can create the `SNOWPACK_MERCHANT_RISK` database and the `SNOWPACK_MERCHANT_RISK_WH` warehouse.

In [ ]:
SCHEMA = "SNOWPACK_MERCHANT_RISK.CORE"
MODEL  = "claude-sonnet-4-5"
import json
from snowflake.snowpark.context import get_active_session
session = get_active_session()
mode = session.sql(f"select mode, note from {SCHEMA}.MERCHANT_RISK_DATA_SOURCE_INFO").collect()[0]
print("data mode:", mode[0], "—", mode[1])

## Step 1 — Explore the benchmark

`MERCHANT_RISK_BENCHMARK` = underwriting applications/decisions by category, credit tier, decision and month, with a monthly CFPB complaint backdrop attached (per-month, non-additive).

In [ ]:
session.sql(f"""
  SELECT category_name, decision,
         SUM(application_count) AS applications,
         ROUND(SUM(risk_score_sum)/NULLIF(SUM(application_count),0),1) AS avg_risk_score
  FROM {SCHEMA}.MERCHANT_RISK_BENCHMARK
  WHERE month_end = (SELECT MAX(month_end) FROM {SCHEMA}.MERCHANT_RISK_BENCHMARK)
  GROUP BY category_name, decision
  ORDER BY category_name, decision
""").to_pandas()

## Step 2 — Cortex Analyst over the semantic view

`MERCHANT_RISK_SV` lets Analyst answer NL questions like *"which categories have the highest decline rate?"*. Direct `SEMANTIC_VIEW()` query below; the agent writes this for you from plain English.

In [ ]:
session.sql(f"""
  SELECT * FROM SEMANTIC_VIEW({SCHEMA}.MERCHANT_RISK_SV
      METRICS applications, approval_rate, avg_risk_score
      DIMENSIONS category_name, credit_tier)
  ORDER BY avg_risk_score DESC
""").to_pandas()

## Step 3 — Verified fact pack

Start-of-window and latest snapshots per (category, credit tier) — the agent's only allowed numbers. Approval rate and risk are aggregated across decisions; the CFPB backdrop is the month's industry figure.

In [ ]:
fact_pack = session.sql(f"""
  WITH w AS (SELECT MIN(month_end) a, MAX(month_end) z FROM {SCHEMA}.MERCHANT_RISK_BENCHMARK),
  agg AS (
    SELECT category_name, credit_tier, month_end,
           SUM(application_count) AS applications,
           ROUND(100.0*SUM(approved_count)/NULLIF(SUM(application_count),0),1) AS approval_rate_pct,
           ROUND(100.0*SUM(declined_count)/NULLIF(SUM(application_count),0),1) AS decline_rate_pct,
           ROUND(SUM(risk_score_sum)/NULLIF(SUM(application_count),0),1) AS avg_risk_score,
           ROUND(SUM(reserve_pct_sum)/NULLIF(SUM(application_count),0),2) AS avg_reserve_pct,
           MAX(cfpb_complaint_count) AS cfpb_complaints,
           MAX(cfpb_pct_monetary_relief) AS cfpb_pct_monetary_relief
    FROM {SCHEMA}.MERCHANT_RISK_BENCHMARK b JOIN w ON b.month_end IN (w.a, w.z)
    GROUP BY category_name, credit_tier, month_end
  )
  SELECT ARRAY_AGG(OBJECT_CONSTRUCT(
           'category', category_name, 'credit_tier', credit_tier, 'month', month_end::string,
           'applications', applications, 'approval_rate_pct', approval_rate_pct,
           'decline_rate_pct', decline_rate_pct, 'avg_risk_score', avg_risk_score,
           'avg_reserve_pct', avg_reserve_pct, 'cfpb_complaints', cfpb_complaints,
           'cfpb_pct_monetary_relief', cfpb_pct_monetary_relief))
  FROM agg
""").collect()[0][0]
print(fact_pack[:700], '...')

## Step 4 — Grounded draft (DATA RULES)

The model may use ONLY the fact pack — no invented numbers, no hedging.

In [ ]:
DATA_RULES = (
    "You are a merchant-underwriting analyst. Write a concise risk note for the processor using ONLY the FACTS JSON.\n"
    "DATA RULES: use only numbers in FACTS, copy exactly; risk_score is 0-100 (higher = riskier); rates are percentages; "
    "dollar fields are USD, express in $B/$M; CFPB figures are an external payment-industry complaint backdrop, NOT this processor's own complaints; "
    "no estimates, no hedging words, no outside benchmarks; missing = n/d.\n"
    "OUTPUT markdown starting with '# ', a one-line takeaway (highest-risk / lowest-approval segments), a table "
    "(category | credit tier | applications | approval % | avg risk | avg reserve %), then 3 bullets (incl. one cross-referencing the CFPB backdrop), ending with '## Data Coverage' (source + window)."
)
prompt = DATA_RULES + f"\n\nFACTS: {fact_pack}"
draft_md = session.sql("SELECT AI_COMPLETE(?, ?)", params=[MODEL, prompt]).collect()[0][0]
print(draft_md)

## Step 5 — Verify against the evidence

A second pass fact-checks every number against the fact pack and flags hedges — the quality gate.

In [ ]:
verify_prompt = (
    "You are a fact-checker. Compare the DRAFT against the FACTS JSON (the only allowed source). "
    "Return ONLY a valid JSON object (double quotes, no code fences, no prose) with keys: status (\"verified\" if every number "
    "and claim is supported by FACTS and there are no hedge words like likely/approximately/around/could, else \"needs_review\"), "
    "residual_hedges (integer), edits (array of short strings naming each unsupported or hedged claim).\n\n"
    f"FACTS: {fact_pack}\n\nDRAFT:\n{draft_md}"
)
raw = session.sql("SELECT AI_COMPLETE(?, ?)", params=[MODEL, verify_prompt]).collect()[0][0]
import re as _re
def _parse_verdict(x):
    t = x.strip()
    for _ in range(3):
        m = _re.match(r"^```[a-zA-Z]*\s*(.*?)\s*```$", t, _re.S)
        if m:
            t = m.group(1).strip()
        try:
            v = json.loads(t)
        except Exception:
            a, b = t.find("{"), t.rfind("}")
            if a >= 0 and b > a:
                try:
                    v = json.loads(t[a:b + 1])
                except Exception:
                    return None
            else:
                return None
        if isinstance(v, str):
            t = v.strip()
            continue
        return v
    return None
verdict = _parse_verdict(raw) or {"status": "needs_review", "residual_hedges": None, "edits": ["unparseable verifier output"]}
status = verdict.get("status", "needs_review")
edits = verdict.get("edits", []) or []
hedges = verdict.get("residual_hedges")
print(f"STATUS: {status.upper()}   residual_hedges={hedges}   edits={len(edits)}")
for e in edits:
    print(" -", e)

## Step 6 — Persist the note

In [ ]:
session.sql(
    f"INSERT INTO {SCHEMA}.MERCHANT_RISK_REPORTS (subject, draft_md, report_md, status, edits_applied, residual_hedges) SELECT ?, ?, ?, ?, ?, ?",
    params=["Merchant risk & underwriting", draft_md, draft_md, status, len(edits), hedges]
).collect()
session.sql(f"SELECT run_id, subject, status, edits_applied, residual_hedges, created_at FROM {SCHEMA}.MERCHANT_RISK_REPORTS ORDER BY created_at DESC LIMIT 5").to_pandas()

## Recap & extensions

You built the verified underwriting loop on free data. `setup.sql` already created a **`MERCHANT_RISK_AGENT`** — chat with it in Snowsight → AI & ML → Agents (*"which categories have the highest decline rate?"*).

**Next:** point `MERCHANT_RISK_APPLICATION` / `MERCHANT_RISK_DECISION` at your own intake and decision data to score your real book. The CFPB backdrop stays as a free external risk signal.

**Try breaking the verifier:** add a made-up claim (e.g., *"approvals will likely rise 10% next quarter"*) to the draft and watch Step 5 flag it *Needs review*.